In [0]:
from datetime import datetime, timedelta

# Get job parameter
run_date_str = dbutils.widgets.get("run_date")
run_date = datetime.strptime(run_date_str, "%Y-%m-%d")

# Subtract 3 days
three_days_before = run_date - timedelta(days=3)
enddate = str(run_date.date())
startDate = str(three_days_before.date())
print(f"Job start: {run_date.date()}")
print(f"3 days before: {three_days_before.date()}")

In [0]:
pip install google-analytics-data

In [0]:
# %restart_python

In [0]:
from google.analytics.data_v1beta import BetaAnalyticsDataClient
from google.analytics.data_v1beta.types import RunReportRequest, DateRange, Dimension, Metric
from google.oauth2 import service_account
import json
import os
from datetime import datetime

In [0]:
# Đường dẫn tới file JSON chứa thông tin xác thực
KEY_PATH = "service_account.json"

# GA4 Property ID (ví dụ: 123456789)
PROPERTY_ID = "373253671"

# Tạo client
credentials = service_account.Credentials.from_service_account_file(KEY_PATH)
client = BetaAnalyticsDataClient(credentials=credentials)

# Tạo request để lấy các chỉ số tương tác
request = RunReportRequest(
    property=f"properties/{PROPERTY_ID}",
    dimensions=[Dimension(name="country"),Dimension(name="countryId"),Dimension(name="date")],
    metrics=[
        Metric(name="activeUsers"),
        Metric(name="screenPageViews"),
        Metric(name="scrolledUsers"),
        Metric(name="sessions"),
        Metric(name="sessionsPerUser"),
        Metric(name="userEngagementDuration")
    ],
    date_ranges=[DateRange(start_date=startDate, end_date=enddate)]
)
response = client.run_report(request)


In [0]:
# Tạo tên file dựa trên PROPERTY_ID, start_date và end_date
start_date = request.date_ranges[0].start_date
end_date = request.date_ranges[0].end_date
filename = f"{PROPERTY_ID}_{start_date}_{end_date}.json"
dbfs_folder_path = "/Workspace/dataflatform_dev/bronze/ga-data"
os.makedirs(dbfs_folder_path, exist_ok=True)

file_path = os.path.join(dbfs_folder_path, filename)

# Xử lý dữ liệu response thành dict
report_data = {
    "headers": {
        "dimensions": [header.name for header in response.dimension_headers],
        "metrics": [header.name for header in response.metric_headers]
    },
    "rows": [
        {
            "dimensions": [value.value for value in row.dimension_values],
            "metrics": [value.value for value in row.metric_values]
        }
        for row in response.rows
    ]
}

# Lưu vào file JSON
with open(file_path, "w", encoding="utf-8") as f:
    json.dump(report_data, f, ensure_ascii=False, indent=2)

print(f"Đã lưu báo cáo vào file {file_path}")

In [0]:
from datetime import datetime

# Đọc nội dung JSON từ file vừa lưu
with open(file_path, "r") as f:
    json_content = f.read()

# Tạo DataFrame với các trường yêu cầu
data = [{
    "processing_time": datetime.now(),
    "file_path": file_path,
    "file_modification_time": datetime.now().date(),
    "content": {"json": json_content}
}]
df = spark.createDataFrame(data)

# Ghi vào bảng Delta
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DateType

schema = StructType([
    StructField("processing_time", TimestampType(), True),
    StructField("file_path", StringType(), True),
    StructField("file_modification_time", DateType(), True),
    StructField("content", StructType([
        StructField("json", StringType(), True)
    ]), True)
])

df = spark.createDataFrame(data, schema=schema)
df.write.format("delta").mode("append").saveAsTable("dataflatform_dev.bronze.ga_event")